In [1]:
import pickle
import os
import scipy

In [13]:
result_path = "/home/jxxiong/A-xjx/deeplde/results/QCQPProblem-200-100-100-10000/method_deeplde/2941ba8910c79199c60ff827cc4523f9a1dbbe4a/1716294108-2732122/"
sol_name = "sol.dict"
sol_path = os.path.join(result_path, sol_name)
sol_path

'/home/jxxiong/A-xjx/deeplde/results/QCQPProblem-200-100-100-10000/method_deeplde/2941ba8910c79199c60ff827cc4523f9a1dbbe4a/1716294108-2732122/sol.dict'

In [14]:
with open(sol_path, 'rb') as f:
    Ytest = pickle.load(f)

In [15]:
method_name = result_path.split("/")[7].split("_")[1]
method_name

'deeplde'

In [16]:
# method_name = "DC3"
# method_name = "EqH_Bis"
# method_name = "NN"

In [17]:
problem_name = result_path.split("/")[6]
problem_name = "_".join(problem_name.split("-")[:-1])
problem_name 

'QCQPProblem_200_100_100'

In [18]:
with open(sol_path, 'rb') as f:
    Ytest = pickle.load(f)

In [19]:
sol_path = os.path.join(result_path, method_name + "_" + problem_name + ".mat")
sol_path

'/home/jxxiong/A-xjx/deeplde/results/QCQPProblem-200-100-100-10000/method_deeplde/2941ba8910c79199c60ff827cc4523f9a1dbbe4a/1716294108-2732122/deeplde_QCQPProblem_200_100_100.mat'

In [20]:
sol = {'x': Ytest}
# save sol to mat file
scipy.io.savemat(sol_path, sol)

In [11]:
sol = scipy.io.loadmat(sol_path)

In [12]:
sol_dc3 = scipy.io.loadmat("/home/jxxiong/A-xjx/deeplde/baseline_sols/DC3_QCQPProblem_100_50_50.mat")
sol_hbis = scipy.io.loadmat("/home/jxxiong/A-xjx/deeplde/baseline_sols/EqH_Bis_QCQPProblem_100_50_50.mat")

In [30]:
sol_dc3

{'__header__': b'MATLAB 5.0 MAT-file Platform: posix, Created on: Wed May 15 15:16:20 2024',
 '__version__': '1.0',
 '__globals__': [],
 'x': array([[ 0.36375588,  0.13743681, -0.13563086, ..., -0.11586664,
         -0.99064706, -0.38261952],
        [ 0.37590876,  0.18431667, -0.17854452, ..., -0.06734392,
         -0.85483829, -0.37102427],
        [ 0.39078339,  0.25218898, -0.20348881, ..., -0.12654611,
         -0.73412756, -0.32168044],
        ...,
        [ 0.39455431,  0.14580834, -0.15475626, ..., -0.08083934,
         -1.06142409, -0.62061411],
        [ 0.37765408,  0.09604769, -0.14589527, ..., -0.03770692,
         -0.94013492, -0.53002713],
        [ 0.41022756,  0.1725261 , -0.16777307, ..., -0.04708973,
         -0.97339668, -0.48437614]])}

In [31]:
sol_hbis

{'__header__': b'MATLAB 5.0 MAT-file Platform: posix, Created on: Tue May 14 20:25:00 2024',
 '__version__': '1.0',
 '__globals__': [],
 'x': array([[ 0.36375588,  0.13743681, -0.13563086, ..., -0.11586664,
         -0.99064706, -0.38261952],
        [ 0.37590876,  0.18431667, -0.17854452, ..., -0.06734392,
         -0.85483829, -0.37102427],
        [ 0.39078339,  0.25218898, -0.20348881, ..., -0.12654611,
         -0.73412756, -0.32168044],
        ...,
        [ 0.39455431,  0.14580834, -0.15475626, ..., -0.08083934,
         -1.06142409, -0.62061411],
        [ 0.37765408,  0.09604769, -0.14589527, ..., -0.03770692,
         -0.94013492, -0.53002713],
        [ 0.41022756,  0.1725261 , -0.16777307, ..., -0.04708973,
         -0.97339668, -0.48437614]])}

## load primal and dual for pdl

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.set_default_dtype(torch.float64)

import operator
from functools import reduce
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import pickle
import time
from setproctitle import setproctitle
import os
import argparse
import sys
import scipy

sys.path.insert(1, os.path.join(sys.path[0], os.pardir))
from utils import my_hash, str_to_bool
import default_args
from method_pdl import Primal_NN, Dual_NN
from qcqp_utils import QCQPProbem

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [113]:
result_path = "/home/jxxiong/A-xjx/deeplde/results/QCQPProblem-100-50-50-10000/method_pdl/b090cb58e70a842975735cbadc72fc6d789673ec/1716216070-3280797/"

with open(os.path.join(result_path, "args.dict"), 'rb') as f:
    args = pickle.load(f)

In [115]:
with open(os.path.join(result_path, "stats.dict"), 'rb') as f:
    stats = pickle.load(f)
    
epoch_stats = {k: stats[k][-1] for k in stats.keys() if "test" in k}
print(' test loss {:.4f}, test eval {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, eq max {:.4f}, eq mean {:.4f}, time {:.4f}'.format(
     np.mean(epoch_stats['test_loss']), np.mean(epoch_stats['test_eval']),
                np.mean(epoch_stats['test_ineq_max']),
                np.mean(epoch_stats['test_ineq_mean']), 
                np.mean(epoch_stats['test_eq_max']), np.mean(epoch_stats['test_eq_mean']), np.mean(epoch_stats['test_time'])))

 test loss -14.6511, test eval -15.3107, ineq max 0.0058, ineq mean 0.0003, eq max 0.0052, eq mean 0.0017, time 0.0001


In [116]:
prob_type = args['probType']
if prob_type == 'simple':
    filepath = os.path.join('/home/jxxiong/A-xjx/deeplde/datasets', 'simple', "random_simple_dataset_var{}_ineq{}_eq{}_ex{}".format(
        args['simpleVar'], args['simpleIneq'], args['simpleEq'], args['simpleEx']))
elif prob_type == 'nonconvex':
    filepath = os.path.join('/home/jxxiong/A-xjx/deeplde/datasets', 'nonconvex', "random_nonconvex_dataset_var{}_ineq{}_eq{}_ex{}".format(
        args['nonconvexVar'], args['nonconvexIneq'], args['nonconvexEq'], args['nonconvexEx']))
elif prob_type == 'convex_qcqp':
    filepath = os.path.join('/home/jxxiong/A-xjx/deeplde/datasets', 'convex_qcqp', "random_{}_{}_dataset_var{}_ineq{}_eq{}_ex{}".format(
        2023, prob_type, args['simpleVar'], args['simpleIneq'], args['simpleEq'], args['simpleEx']))
    with open(filepath, 'rb') as f:
        dataset = pickle.load(f)
    data = QCQPProbem(dataset, 833)
    data.device = DEVICE
    for attr in dir(data):
        var = getattr(data, attr)
        if torch.is_tensor(var):
            try:
                setattr(data, attr, var.to(DEVICE))
            except AttributeError:
                pass

if prob_type != 'convex_qcqp':
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    for attr in dir(data):
        var = getattr(data, attr)
        if not callable(var) and not attr.startswith("__") and torch.is_tensor(var):
            try:
                setattr(data, attr, var.to(DEVICE))
            except AttributeError:
                pass
    data._device = DEVICE

In [117]:
test_dataset = TensorDataset(data.testX)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset))

In [118]:
primal = Primal_NN(data, args)
primal.load_state_dict(torch.load(os.path.join(result_path, "primal_net.dict")))
primal.eval()
primal.to(DEVICE)

dual = Dual_NN(data, args)
dual.eval()
dual.load_state_dict(torch.load(os.path.join(result_path, "dual_net.dict")))
dual.to(DEVICE)

Dual_NN(
  (net): Sequential(
    (0): Linear(in_features=50, out_features=500, bias=True)
    (1): ReLU()
    (2): Linear(in_features=500, out_features=500, bias=True)
    (3): ReLU()
    (4): Linear(in_features=500, out_features=100, bias=True)
  )
)

In [119]:
for Xtest in test_loader:
    Xtest = Xtest[0].to(DEVICE)                
    Ytest_pred = primal(Xtest)
    dual_pred = dual(Xtest)

In [120]:
Ytest_pred.shape

torch.Size([833, 100])

In [121]:
dual_pred.shape

torch.Size([833, 100])

In [122]:
dual_pred

tensor([[ 0.0000e+00,  1.7153e-03,  0.0000e+00,  ..., -5.4247e-01,
         -6.1054e-02,  6.5725e-01],
        [ 0.0000e+00, -2.5425e-03,  0.0000e+00,  ...,  2.2432e-03,
         -6.8850e-01, -8.5091e-01],
        [ 0.0000e+00, -3.0243e-03,  0.0000e+00,  ..., -8.6854e-01,
         -7.3020e-01,  4.9930e-01],
        ...,
        [ 0.0000e+00, -5.5277e-03,  0.0000e+00,  ..., -8.3480e-01,
         -2.8151e-01, -5.5341e-02],
        [ 0.0000e+00,  1.4481e-03,  0.0000e+00,  ..., -1.3275e+00,
          8.4324e-02, -2.6743e-01],
        [ 0.0000e+00,  1.1628e-03,  0.0000e+00,  ..., -6.1315e-01,
          1.7735e-01,  2.5482e-01]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [123]:
data = {'x': Ytest_pred.detach().cpu().numpy(), 
        'mu': dual_pred.detach().cpu().numpy()}

In [124]:
method_name = "pdl"

In [125]:
problem_name = result_path.split("/")[6]
problem_name = "_".join(problem_name.split("-")[:-1])
problem_name 

'QCQPProblem_100_50_50'

In [126]:
sol_path = os.path.join(result_path, method_name + "_" + problem_name + ".mat")
sol_path

'/home/jxxiong/A-xjx/deeplde/results/QCQPProblem-100-50-50-10000/method_pdl/b090cb58e70a842975735cbadc72fc6d789673ec/1716216070-3280797/pdl_QCQPProblem_100_50_50.mat'

In [127]:
scipy.io.savemat(sol_path, data)

In [21]:
sol = scipy.io.loadmat(sol_path)
sol.keys()

dict_keys(['__header__', '__version__', '__globals__', 'x', 'mu'])

# DC3

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
torch.set_default_dtype(torch.float64)

import operator
from functools import reduce
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import pickle
import time
from setproctitle import setproctitle
import os
import argparse
import sys
import scipy

sys.path.insert(1, os.path.join(sys.path[0], os.pardir))
from utils import my_hash, str_to_bool
import default_args
from method import *
from qcqp_utils import QCQPProbem

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [28]:
result_path = "/home/jxxiong/A-xjx/deeplde/results/QCQPProblem-200-100-100-10000/method copy/1abcb453d00261755541ec7f74d8cca14caeb58e/1716293791-3563476/"

with open(os.path.join(result_path, "args.dict"), 'rb') as f:
    args = pickle.load(f)

In [29]:
with open(os.path.join(result_path, "stats.dict"), 'rb') as f:
    stats = pickle.load(f)
    
epoch_stats = {k: stats[k][-1] for k in stats.keys() if "test" in k}
print(' test loss {:.4f}, test eval {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, eq max {:.4f}, eq mean {:.4f}, time {:.4f}'.format(
     np.mean(epoch_stats['test_loss']), np.mean(epoch_stats['test_eval']),
                np.mean(epoch_stats['test_ineq_max']),
                np.mean(epoch_stats['test_ineq_mean']), 
                np.mean(epoch_stats['test_eq_max']), np.mean(epoch_stats['test_eq_mean']), np.mean(epoch_stats['test_time'])))

 test loss -35.7400, test eval -35.7413, ineq max 0.0003, ineq mean 0.0000, eq max 0.0000, eq mean 0.0000, time 6.9510


In [30]:
prob_type = args['probType']
if prob_type == 'simple':
    filepath = os.path.join('/home/jxxiong/A-xjx/deeplde/datasets', 'simple', "random_simple_dataset_var{}_ineq{}_eq{}_ex{}".format(
        args['simpleVar'], args['simpleIneq'], args['simpleEq'], args['simpleEx']))
elif prob_type == 'nonconvex':
    filepath = os.path.join('/home/jxxiong/A-xjx/deeplde/datasets', 'nonconvex', "random_nonconvex_dataset_var{}_ineq{}_eq{}_ex{}".format(
        args['nonconvexVar'], args['nonconvexIneq'], args['nonconvexEq'], args['nonconvexEx']))
elif prob_type == 'convex_qcqp':
    filepath = os.path.join('/home/jxxiong/A-xjx/deeplde/datasets', 'convex_qcqp', "random_{}_{}_dataset_var{}_ineq{}_eq{}_ex{}".format(
        2023, prob_type, args['simpleVar'], args['simpleIneq'], args['simpleEq'], args['simpleEx']))
    with open(filepath, 'rb') as f:
        dataset = pickle.load(f)
    data = QCQPProbem(dataset, 833)
    data.device = DEVICE
    for attr in dir(data):
        var = getattr(data, attr)
        if torch.is_tensor(var):
            try:
                setattr(data, attr, var.to(DEVICE))
            except AttributeError:
                pass

if prob_type != 'convex_qcqp':
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    for attr in dir(data):
        var = getattr(data, attr)
        if not callable(var) and not attr.startswith("__") and torch.is_tensor(var):
            try:
                setattr(data, attr, var.to(DEVICE))
            except AttributeError:
                pass
    data._device = DEVICE

In [31]:
test_dataset = TensorDataset(data.testX)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset))

In [32]:
solver_net = NNSolver(data, args)
solver_net.to(DEVICE)
solver_net.load_state_dict(torch.load(os.path.join(result_path, "solver_net.dict")))

<All keys matched successfully>

In [33]:
solver_net.eval()
for Xtest in test_loader:
    Xtest = Xtest[0].to(DEVICE)
    Ytest = solver_net(Xtest)
    Ycorr, steps = grad_steps_all(data, Xtest, Ytest, args)

In [34]:
data = {'x': Ycorr.detach().cpu().numpy()}

In [35]:
method_name = "DC3"
problem_name = result_path.split("/")[6]
problem_name = "_".join(problem_name.split("-")[:-1])
print(problem_name)
sol_path = os.path.join(result_path, method_name + "_" + problem_name + ".mat")
sol_path

QCQPProblem_200_100_100


'/home/jxxiong/A-xjx/deeplde/results/QCQPProblem-200-100-100-10000/method copy/1abcb453d00261755541ec7f74d8cca14caeb58e/1716293791-3563476/DC3_QCQPProblem_200_100_100.mat'

In [36]:
scipy.io.savemat(sol_path, data)